# Qwen3.5-0.8B Vision → LaTeX OCR, as a Nawāt run

The [Unsloth vision notebook](https://unsloth.ai/docs/basics/vision-fine-tuning)
for handwritten-formula OCR, with the storage handled by Nawāt.

What is different from the original, and nothing else is:

| Unsloth notebook | Here |
| --- | --- |
| `from_pretrained("unsloth/Qwen3.5-0.8B")` | `from_pretrained(run.model_dir)` |
| `load_dataset("unsloth/LaTeX_OCR")` | `load_dataset(run.dataset_dir)` |
| `output_dir = "outputs"` | `output_dir = run.scratch_dir("trainer")` |
| `save_pretrained("qwen_lora")` | `save_pretrained(run.artifact_dir("adapter"))` |
| `save_pretrained_gguf("qwen_finetune", ...)` | `save_pretrained_gguf(str(run.artifact_dir("gguf")), ...)` |
| `push_to_hub(..., token=...)` | `run.finish()` — verified publish to your own store |

The model and the dataset come from local cache, then object storage, then
Hugging Face — in that order, and the internet at most once ever. While the run
is open they are leased to this kernel, so nothing can evict them between cells.
When you call `run.finish()` the adapter is uploaded, verified file by file, and
reclaimed from local disk, and the whole thing is on the record in `nawat runs`.

> Every cell below also runs unchanged as a submitted script — see
> `train_latex_ocr.py` next to this notebook, and the last section.

---

*Derived from the Unsloth vision fine-tuning notebook and, like it, licensed
[LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)
— not under the noncommercial licence covering the rest of this repository. See
[LICENSE](LICENSE) in this directory.*

### Installation, once per machine

The stock Unsloth notebook reinstalls on every run, because Colab throws the
machine away between sessions. On a workstation that is pure waste — several
minutes and a few gigabytes of downloads to arrive back where you started.

So the whole block below is guarded: if `unsloth` already imports, nothing runs
at all. When it does run, `uv` resolves and links from a persistent cache
(`UV_CACHE_DIR`, `~/.cache/uv` by default), so even a rebuilt environment mostly
hardlinks from local disk instead of going out to the network.

Set `NAWAT_FORCE_REINSTALL=1` to override the guard and upgrade deliberately.

In [ ]:
%%capture
import importlib.util, os

# One cache for every environment on this machine — the point of using uv here.
os.environ.setdefault("UV_CACHE_DIR", os.path.expanduser("~/.cache/uv"))

INSTALLED = importlib.util.find_spec("unsloth") is not None
FORCE     = os.environ.get("NAWAT_FORCE_REINSTALL") == "1"
COLAB     = "COLAB_" in "".join(os.environ.keys())

if not INSTALLED or FORCE:
    !pip install --upgrade -qqq uv
    if importlib.util.find_spec("torch") is None or COLAB:
        try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
        except: _numpy = "numpy"; _pil = "pillow"
        !uv pip install -qqq \
            "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
            "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
            "unsloth[base] @ git+https://github.com/unslothai/unsloth"
        !uv pip install -qqq --no-deps "torchcodec==0.7.0"
    else:
        !uv pip install -qqq unsloth
    !uv pip install -qqq --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
    !uv pip install -qqq transformers==5.2.0
    # causal_conv1d is supported only on torch==2.8.0. On newer torch this builds slowly.
    !uv pip install -qqq --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
    import torch
    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
        !uv pip install -qqq --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"

# Nawāt itself — small, pure Python, and equally skippable once present:
if importlib.util.find_spec("nawat") is None:
    !uv pip install -qqq "nawat[notebook] @ git+https://github.com/murtadha-lap/nawat.git"

In [ ]:
# Tilelang is unavailable below compute capability 8.0; the flag must be set in
# this process whether or not the install block above ran.
import torch
if not (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8):
    import os; os.environ["FLA_TILELANG"] = "0"

import importlib.metadata as md
for package in ("unsloth", "torch", "transformers", "trl", "nawat"):
    try:
        print(f"{package:14} {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package:14} not installed")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

### Will it fit, and is there room?

Before spending GPU hours. `nawat estimate` reports VRAM against this host's
card; `nawat status` reports the cache against its ceiling. Neither downloads
anything.

In [ ]:
!nawat status
!nawat estimate --model models/unsloth/Qwen3.5-0.8B --method lora --bits 16 --batch 2 --seq 2048

### Open the run

This stages both inputs — cache, then object storage, then Hugging Face — and
holds them under a lease keyed to this kernel's pid. The lease dies with the
kernel, crash included, so a forgotten notebook cannot wedge the cache and a
live one cannot lose its weights to an eviction.

`params` are the knobs of the experiment. They are recorded with the run, and
`nawat submit --param max_steps=60` overrides them when this notebook graduates
into a script — which is why every value below is read back through
`run.param()` rather than written inline.

In [ ]:
import nawat

run = nawat.begin_run(
    model   = "models/unsloth/Qwen3.5-0.8B",     # the tail IS the Hugging Face repo id
    dataset = "datasets/unsloth/LaTeX_OCR",
    params  = {"max_steps": 30, "learning_rate": 2e-4, "rank": 16},
    notes   = "LaTeX OCR baseline, vision + language layers",
)
run

In [ ]:
print("model  ", run.model_dir)
print("dataset", run.dataset_dir)
print("outputs", run.out_dir)

### Unsloth

From here on this is the stock notebook.

In [ ]:
from unsloth import FastVisionModel  # FastLanguageModel for LLMs
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    run.model_dir,                    # ← was "unsloth/Qwen3.5-0.8B"
    load_in_4bit = False,             # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth",  # True or "unsloth" for long context
)

We now add LoRA adapters for parameter efficient finetuning - this allows us to
only efficiently train 1% of all parameters.

We also support finetuning ONLY the vision part of the model, or ONLY the
language part. Or you can select both! You can also select to finetune the
attention or the MLP layers!

In [ ]:
rank = run.param("rank", 16)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,  # False if not finetuning vision layers
    finetune_language_layers   = True,  # False if not finetuning language layers
    finetune_attention_modules = True,  # False if not finetuning attention layers
    finetune_mlp_modules       = True,  # False if not finetuning MLP layers

    r = rank,             # The larger, the higher the accuracy, but might overfit
    lora_alpha = rank,    # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

### Data Prep

A sampled dataset of handwritten maths formulas. The goal is to convert these
images into a computer readable form — LaTeX — so we can render it.

`load_dataset` is pointed at the staged directory rather than a repo id. Nothing
is downloaded here: the hub is switched off for the life of the run, so a typo
in a dataset name fails loudly instead of quietly pulling gigabytes onto a disk
that has no room for them.

In [ ]:
from datasets import load_dataset

dataset = load_dataset(run.dataset_dir, split = "train")   # ← was "unsloth/LaTeX_OCR"
dataset

In [ ]:
dataset[2]["image"]

In [ ]:
dataset[2]["text"]

We can also render the LaTeX in the browser directly!

In [ ]:
from IPython.display import display, Math, Latex

latex = dataset[2]["text"]
display(Math(latex))

To format the dataset, all vision finetuning tasks should be formatted as follows:

```python
[
{ "role": "user",
  "content": [{"type": "text",  "text": Q}, {"type": "image", "image": image} ]
},
{ "role": "assistant",
  "content": [{"type": "text",  "text": A} ]
},
]
```

In [ ]:
instruction = "Write the LaTeX representation for this image."

def convert_to_conversation(sample):
    conversation = [
        { "role": "user",
          "content" : [
            {"type" : "text",  "text"  : instruction},
            {"type" : "image", "image" : sample["image"]} ]
        },
        { "role" : "assistant",
          "content" : [
            {"type" : "text",  "text"  : sample["text"]} ]
        },
    ]
    return { "messages" : conversation }
pass

In [ ]:
converted_dataset = [convert_to_conversation(sample) for sample in dataset]
converted_dataset[0]

Let's first see before we do any finetuning what the model outputs for the third example!

In [ ]:
FastVisionModel.for_inference(model)  # Enable for inference!

image = dataset[2]["image"]
instruction = "Write the LaTeX representation for this image."

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

### Train the model

Two changes from the stock trainer setup, both about where bytes land:

- `callbacks = [run.callback()]` streams loss, learning rate, grad norm and
  steps/second into the run's metric series — `nawat metrics <id> -f` from any
  other shell, and the series outlives the weights.
- `output_dir = run.scratch_dir("trainer")` keeps intermediate checkpoints out
  of `run.out_dir`. Anything under `out_dir` is published as an artifact; a
  directory of checkpoints is not an artifact, it is scratch, and this deletes
  it at the end.

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)  # Enable for training!

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),  # Must use!
    train_dataset = converted_dataset,
    callbacks = [run.callback()],          # ← the live loss trace
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps     = run.param("max_steps", 30),
        learning_rate = run.param("learning_rate", 2e-4),
        # num_train_epochs = 1,  # Set this instead of max_steps for full training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = str(run.scratch_dir("trainer")),   # ← was "outputs"
        report_to = "none",

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    ),
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

### The loss trace

`run.callback()` has been writing every logging step to the run's durable
series. It is a file beside the run log — never in the cache — so this plot
renders identically weeks later, long after the weights have been reclaimed.

In [ ]:
import matplotlib.pyplot as plt

series = nawat.trace(run.id)
loss = series["loss"]

plt.figure(figsize=(7, 3))
plt.plot([p["step"] for p in loss], [p["value"] for p in loss])
plt.xlabel("step"); plt.ylabel("loss"); plt.title(f"run {run.id}")
plt.show()

# Or from any other shell, live:  nawat metrics <id> -f

### Inference

Let's run the model! You can change the instruction and input - leave the output
blank!

We use `min_p = 0.1` and `temperature = 1.5`. Read this
[Tweet](https://x.com/menhguin/status/1826132708508213629) for more information
on why.

In [ ]:
FastVisionModel.for_inference(model)  # Enable for inference!

image = dataset[2]["image"]
instruction = "Write the LaTeX representation for this image."

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

### Saving the adapter

`run.artifact_dir(name)` is a directory that becomes `runs/<id>/<name>` when the
run finishes. Save into it exactly as you would into any local folder.

The adapter alone is a few hundred megabytes against a multi-gigabyte base, which
is the whole reason to train this way.

In [ ]:
adapter = run.artifact_dir("adapter")         # ← was "qwen_lora"
model.save_pretrained(adapter)
tokenizer.save_pretrained(adapter)

sorted(p.name for p in adapter.iterdir())

### Merged FP16, for anything that cannot load an adapter

A LoRA is a patch; some runtimes want the whole weight matrix. `merged` is that —
base plus adapter, folded into one FP16 checkpoint, roughly the size of the base
model. It is also the input GGUF conversion needs.

Skip this cell if you only ever serve through vLLM: the next-but-one section
hot-loads the adapter onto the base without merging anything.

In [ ]:
merged = run.artifact_dir("merged")
model.save_pretrained_merged(str(merged), tokenizer)   # str: Unsloth joins paths as text

sum(p.stat().st_size for p in merged.rglob("*") if p.is_file()) / 1e9  # GB

### GGUF, for llama.cpp and Ollama

`save_pretrained_gguf` clones and builds llama.cpp on first use, then converts
the merged weights. Some methods from the Unsloth docs:

- `q8_0` — fast conversion, high resource use, generally acceptable
- `q4_k_m` — recommended. Q6_K for half of `attention.wv` and `feed_forward.w2`, else Q4_K
- `q5_k_m` — recommended. Q6_K for the same halves, else Q5_K
- `f16` — no quantization, just the format

This is the step most likely to fail, and it is worth knowing why before you
spend the disk on it: llama.cpp has to recognise the architecture, and support
for vision encoders lags well behind support for text models. If the conversion
refuses, that is llama.cpp's answer and not something Nawāt can paper over — the
adapter and merged export are already saved, the failure is in the run log, and
`run.finish()` still publishes everything that did succeed.

In [ ]:
gguf = run.artifact_dir("gguf")

try:
    model.save_pretrained_gguf(
        str(gguf), tokenizer,
        quantization_method = "q4_k_m",       # or "q8_0", "q5_k_m", "f16"
    )
    print("converted:", sorted(p.name for p in gguf.rglob("*.gguf")))
except Exception as exc:
    # Usually: llama.cpp does not support this architecture yet.
    run.log(f"gguf conversion failed: {type(exc).__name__}: {exc}")
    print("GGUF conversion failed, continuing without it:", exc)

Several quantizations at once is much faster than one at a time — llama.cpp is
built and the FP16 intermediate produced only once:

```python
model.save_pretrained_gguf(
    str(run.artifact_dir("gguf")), tokenizer,
    quantization_method = ["q4_k_m", "q5_k_m", "q8_0"],
)
```

Each one is a file inside `runs/<id>/gguf`, so they publish, verify and reclaim
as a single artifact.

### Finish the run

This is the whole point. `finish()` uploads every directory under `out_dir` as
its own artifact — `adapter`, `merged`, `gguf`, whatever is there — verifies each
in object storage file by file, reclaims the local copy, and closes the record.
Then it releases the model and dataset leases, so the cache is free to evict them
when it next needs room.

Nothing is deleted that has not been verified present in object storage first.
If the store cannot be reached, you get a refusal that names the reason and a
full disk, rather than a gamble.

Note what this means for the merged and GGUF exports specifically: they are the
files that silently fill a disk on an ordinary setup, because nothing ever
decides they can go. Here they are uploaded and reclaimed the moment the run
ends, and they come back on demand.

In [ ]:
record = run.finish()

print(record.state.value, "·", record.id)
for key in record.artifacts:
    print(" ", key)

In [ ]:
# Everything is now in object storage and off the local disk:
!nawat ls
!nawat run {run.id}

### Using the GGUF later

The file is in object storage, not on this disk. `nawat resolve` brings it back —
making room first if the ceiling requires it — and prints where it landed:

```bash
nawat resolve runs/<id>/gguf         # prints the local path
ls "$(nawat resolve runs/<id>/gguf)"

llama-cli -m "$(nawat resolve runs/<id>/gguf)"/*.gguf -p "..."
```

To promote one to a named deployment artifact, separate from the run that
happened to produce it:

```bash
nawat publish "$(nawat resolve runs/<id>/gguf)" exports/latex-ocr-q4
nawat keep exports/latex-ocr-q4       # exempt it from reclamation
```

For Ollama, point a Modelfile at the resolved path:

```bash
printf 'FROM %s\n' "$(nawat resolve exports/latex-ocr-q4)"/*.gguf > Modelfile
ollama create latex-ocr -f Modelfile
```

### Test the adapter without merging it

A ~200 MB LoRA against a multi-GB base: merging to test would multiply the
storage cost per experiment by two orders of magnitude. Serve the base once and
hot-load the adapter onto it instead — seconds, no merge, no restart.

```bash
nawat serve models/unsloth/Qwen3.5-0.8B        # stages weights, starts vLLM
nawat adapter runs/<id>/adapter --name latex-ocr

curl http://127.0.0.1:8001/v1/chat/completions -d '{
  "model": "latex-ocr",
  "messages": [{"role": "user", "content": [
    {"type": "image_url", "image_url": {"url": "data:image/png;base64,..."}},
    {"type": "text", "text": "Write the LaTeX representation for this image."}]}]
}'

nawat session --stop     # or walk away: idle teardown releases GPU and disk
```

And to score it against a held-out set — character and word error rate, written
into the run record — once you have one as JSONL, one sample per line with a
`prompt`, a `reference` transcription and an `image` path relative to the
dataset, published under a key of its own:

```bash
nawat eval <id> --data datasets/latex-ocr-holdout --limit 200
```

### Graduating to `nawat submit`

Everything above was written through `run.model_dir`, `run.param(...)` and
`run.artifact_dir(...)`. The module-level equivalents — `nawat.model_dir()`,
`nawat.param(...)`, `nawat.artifact_dir(...)` — read the environment when the
executor set one and the open run otherwise, so the same cells run either way.

`train_latex_ocr.py` beside this notebook is exactly that: these cells, with
`begin_run`/`finish` removed because the executor does both. Queue it, walk
away, and watch it from anywhere:

```bash
nawat submit train_latex_ocr.py \
  --model   models/unsloth/Qwen3.5-0.8B \
  --dataset datasets/unsloth/LaTeX_OCR \
  --param   max_steps=500 --param learning_rate=2e-4 \
  --notes   "LaTeX OCR, long run"

nawat logs    <id> -f
nawat metrics <id> -f
```

The notebook is for deciding what to run. `nawat submit` is for running it for
six hours without a browser tab open.

You can also submit this notebook itself — `nawat submit latex_ocr_qwen3_5_vision.ipynb
--model ... --dataset ...` — in which case the executor runs it with nbconvert
and archives the executed copy as the run record. Delete the `begin_run` and
`finish` cells first: under the executor the run already exists.